# 1. Setup
DINO + frozen CLIP text alignment. Based on ToTnki: two independent frozen timm DINO towers, same common seed/data and20-epoch FP32 training recipe. No old user checkpoint is loaded. Only common/ and installed libraries are required. B needs transformers==4.52.2 and huggingface_hub<1.


In [ ]:
import copy, gc, json, math, sys
from pathlib import Path
from datetime import datetime
import timm, torch
import pandas as pd
from torch import nn
from torch.nn import functional as F
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'common/dataset.py').is_file())
sys.path.insert(0, str(ROOT))
from common import config as cfg
from common.dataset import get_loaders, get_pairs, get_crop_and_disease_names, load_split, get_final_test_loaders
from common.evaluate import evaluate, val_accuracy, concept_accuracy
MODEL = 'vit_base_patch14_dinov2.lvd142m'
MODEL_NAME, FAMILY = 'approach3b_dino_text_alignment', 'B'
DEBUG = True
HP = dict(backbone=MODEL, batch_size=8 if DEBUG else 16, epochs=1 if DEBUG else 20,
          crop_lr=1e-4, disease_lr=3e-4, weight_decay=1e-4, label_smoothing=0.1,
          seed=cfg.SEED, norm='imagenet', hidden_dim=256, feature_dim=128,
          supcon_temperature=0.1, text_batch_size=16, dino_revision='main', text_revision='main',
          score_grid=[[0.5,0.25,0.25]],
          pair_templates=['a photo of a {crop} leaf with {disease}',
                          'a field photo of {crop} showing {disease} symptoms',
                          'a close-up leaf of {crop} affected by {disease}'],
          healthy_templates=['a photo of a healthy {crop} leaf','a field photo of a healthy {crop} plant'])
EPOCHS = HP['epochs']
cfg.NUM_WORKERS = 0  # Exact ToTnki runtime setting; common files are unchanged.
cfg.set_seed(cfg.SEED)
RUN = MODEL_NAME + ('_debug' if DEBUG else '') + '_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f')
OUT = cfg.CHECKPOINT_DIR / RUN
environment = dict(python=sys.version, torch=str(torch.__version__), timm=str(timm.__version__), device=str(cfg.DEVICE))

if not DEBUG:
    assert (HP['epochs'],HP['batch_size'],HP['crop_lr'],HP['disease_lr'],HP['weight_decay'],HP['label_smoothing']) == (20,16,1e-4,3e-4,1e-4,0.1), 'Keep the ToTnki baseline training budget.'


# 2. Data
Direct common loaders: formal7223 train/777 validation,323 development pairs. DEBUG uses common subsets. No extra holdout, sampler or augmentation. No official test CSV is read here.


In [ ]:
# Same common data path as the old CLIP notebook; DINO uses imagenet normalization.
frames = [load_split(cfg.TRAIN_CSV)]
assert len(frames[0]) == 8000
pairs = get_pairs()
assert pairs.shape == (cfg.NUM_SEEN_PAIRS, 2)
concept_names = get_crop_and_disease_names()
def loaders():
    cfg.set_seed(cfg.SEED)
    return get_loaders(HP['batch_size'], norm=HP['norm'], debug=DEBUG)
train_loader, val_loader, seen_loader, unseen_loader = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
assert not set(train_loader.dataset.frame.image) & set(val_loader.dataset.frame.image)
sample_counts = dict(train=len(train_loader.dataset), val=len(val_loader.dataset))
print('Device:', cfg.DEVICE, 'Data:', cfg.DATA_DIR, 'Samples:', sample_counts)


# 3. Model
Frozen DINO dual towers aligned with frozen CLIP text embeddings; no CLIP image encoder.


In [ ]:
def snapshot(repository,revision,allow_download,patterns):
    from huggingface_hub import snapshot_download
    path=Path(snapshot_download(repository,revision=revision,allow_patterns=patterns,local_files_only=not allow_download))
    return path

class Dino(nn.Module):
    """Exact two-independent-tower initialization from ToTnki's released runner."""
    def __init__(self,config,device,allow_download=False,revision=None):
        super().__init__()
        import timm
        path=snapshot('timm/vit_base_patch14_dinov2.lvd142m',revision or config['dino_revision'],allow_download,
                      ['config.json','model.safetensors','pytorch_model.bin'])
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        def pinned_source_model(classes):
            # Same source factory, with only an explicit weight-file binding added.
            model=timm.create_model(MODEL,pretrained=True,num_classes=classes,img_size=cfg.IMAGE_SIZE,
                                    pretrained_cfg_overlay={'file':str(weights)})
            model.requires_grad_(False);model.head.requires_grad_(True)
            return model
        self.crop=pinned_source_model(cfg.NUM_CROPS)
        self.disease=pinned_source_model(cfg.NUM_DISEASES)
        self.augmentation_rng_state=torch.get_rng_state().clone()
        self.width=self.crop.num_features
        weights=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='timm/vit_base_patch14_dinov2.lvd142m',revision=path.name,
            architecture=MODEL,width=self.width,independent_visual_towers=2,
            initialization='ToTnki source factory settings, crop then disease, explicitly pinned public weight file',
            preprocessing='common imagenet; img_size224; timm native position resampling and pre_logits')
        self.to(device)
    def policy(self,blocks,checkpointing=True):
        if blocks!=0:raise ValueError('ToTnki baseline freezes both visual towers; keep partial_blocks=0.')
        self.requires_grad_(False)
    def features(self,pixels):
        return (self.crop.forward_head(self.crop.forward_features(pixels),pre_logits=True).float(),
                self.disease.forward_head(self.disease.forward_features(pixels),pre_logits=True).float())
    def forward(self,pixels):return self.features(pixels)

class TextOnly:
    """No CLIPModel, CLIPVisionModel, OpenCLIP model, or image encoder is created."""
    def __init__(self,config,device,allow_download=False,revision=None):
        from transformers import CLIPTextModelWithProjection, CLIPTokenizerFast
        path=snapshot('openai/clip-vit-base-patch16',revision or config['text_revision'],allow_download,
            ['*.json','vocab.json','merges.txt','model.safetensors','pytorch_model.bin'])
        self.model=CLIPTextModelWithProjection.from_pretrained(path,local_files_only=True).to(device).eval().requires_grad_(False)
        self.tokenizer=CLIPTokenizerFast.from_pretrained(path,local_files_only=True)
        if any('vision' in name for name,_ in self.model.named_modules()):raise RuntimeError('A CLIP vision encoder was instantiated.')
        self.dimension=self.model.config.projection_dim;self.device=device
        if self.dimension!=512:raise ValueError('Expected the CLIP B/16 512-dimensional text projection.')
        weight=next(path/name for name in ['model.safetensors','pytorch_model.bin'] if (path/name).is_file())
        self.metadata=dict(repository='openai/clip-vit-base-patch16',revision=path.name,
            class_name=type(self.model).__name__,dimension=self.dimension,trainable_parameters=0,clip_image_encoder=False)
    @torch.no_grad()
    def encode(self,prompts,size):
        self.assert_frozen();result=[]
        for start in range(0,len(prompts),size):
            tokens=self.tokenizer(prompts[start:start+size],padding=True,truncation=True,max_length=77,return_tensors='pt').to(self.device)
            result.append(F.normalize(self.model(**tokens).text_embeds.float(),dim=1))
        return torch.cat(result)
    def assert_frozen(self):
        if self.model.training or any(p.requires_grad or p.grad is not None for p in self.model.parameters()):
            raise RuntimeError('CLIP text encoder must be eval, frozen and gradient-free.')

def text_banks(text,pairs,names,config,ensemble=False):
    crops,diseases=names
    def clean(x):return x.replace('_',' ')
    pair_groups=[]
    for c,d in pairs.tolist():
        if clean(diseases[d]).strip().casefold()=='healthy':
            templates=config['healthy_templates'] if ensemble else config['healthy_templates'][:1]
        else:templates=config['pair_templates'] if ensemble else config['pair_templates'][:1]
        pair_groups.append([t.format(crop=clean(crops[c]),disease=clean(diseases[d])) for t in templates])
    flat=[p for group in pair_groups for p in group];encoded=text.encode(flat,config['text_batch_size'])
    bank=[];offset=0
    for group in pair_groups:
        bank.append(F.normalize(encoded[offset:offset+len(group)].mean(0),dim=0));offset+=len(group)
    crop_prompts=[f'a photo of a {clean(crops[i])} plant' for i in range(cfg.NUM_CROPS)]
    disease_prompts=[('a photo of a healthy leaf' if clean(diseases[i]).casefold()=='healthy' else f'a leaf showing {clean(diseases[i])}') for i in range(cfg.NUM_DISEASES)]
    return [torch.stack(bank),text.encode(crop_prompts,config['text_batch_size']),text.encode(disease_prompts,config['text_batch_size'])],dict(pair=pair_groups,crop=crop_prompts,disease=disease_prompts)

def projection(width,hidden,out):return nn.Sequential(nn.Linear(width,hidden),nn.ReLU(),nn.Linear(hidden,out))

class Model(nn.Module):
    def __init__(self,backbone,family,config,text_dimension=512,banks=None):
        super().__init__();self.backbone=backbone;self.family=family
        width=backbone.width;hidden=config['hidden_dim'];dim=config['feature_dim'] if family=='A' else text_dimension
        self.crop_projection=projection(width,hidden,dim);self.disease_projection=projection(width,hidden,dim)
        self.crop_head=nn.Linear(dim,cfg.NUM_CROPS);self.disease_head=nn.Linear(dim,cfg.NUM_DISEASES)
        if family=='B':
            self.pair_projection=projection(width,hidden,dim)
            self.log_temperature=nn.Parameter(torch.tensor(math.log(1/.07)))
            for i,bank in enumerate(banks or []):self.register_buffer('text'+str(i),bank.detach())
    def install_text(self,banks):
        for i,bank in enumerate(banks):setattr(self,'text'+str(i),bank.detach())
    def forward(self,pixels,aligned=False):
        h=self.backbone(pixels)
        hc,hd=h if isinstance(h,tuple) else (h,h)
        crop=F.normalize(self.crop_projection(hc).float(),dim=1)
        disease=F.normalize(self.disease_projection(hd).float(),dim=1)
        if getattr(self,'original_baseline',False):
            return None,self.backbone.crop.head(hc),self.backbone.disease.head(hd),crop,disease
        if self.family=='A':return None,self.crop_head(crop),self.disease_head(disease),crop,disease
        pair=F.normalize(self.pair_projection((hc+hd)/2).float(),dim=1)
        if not aligned:
            return None,self.crop_head((crop+pair)/2),self.disease_head((disease+pair)/2),crop,disease
        scale=self.log_temperature.clamp(math.log(1.),math.log(100.)).exp()
        return scale*pair@self.text0.T,scale*crop@self.text1.T,scale*disease@self.text2.T,crop,disease

def nonsemantic_banks(pairs,device,primitive=None):
    if primitive is None:
        generator=torch.Generator().manual_seed(cfg.SEED)
        crop=F.normalize(torch.randn(cfg.NUM_CROPS,512,generator=generator),dim=1).to(device)
        disease=F.normalize(torch.randn(cfg.NUM_DISEASES,512,generator=generator),dim=1).to(device)
    else:crop,disease=primitive
    rows=pairs.to(device)
    pair=F.normalize(crop[rows[:,0]]+disease[rows[:,1]],dim=1)
    return [pair,crop,disease]

def positive_mask(crops, diseases, cross_crop=True):
    result = diseases[:, None] == diseases[None, :]
    result &= ~torch.eye(len(crops), dtype=torch.bool, device=crops.device)
    if cross_crop: result &= crops[:, None] != crops[None, :]
    return result

def supcon(features, crops, diseases, temperature=.1, cross_crop=True):
    features = F.normalize(features.float(), dim=1)
    positives = positive_mask(crops, diseases, cross_crop)
    valid = positives.any(1)
    if not valid.any(): return features.sum()*0, 0.
    diagonal = torch.eye(len(features), dtype=torch.bool, device=features.device)
    logits = (features @ features.T / temperature).masked_fill(diagonal, -torch.inf)
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)
    # masked_fill avoids 0 * -inf on the diagonal.
    means = log_prob.masked_fill(~positives, 0).sum(1) / positives.sum(1).clamp_min(1)
    return -means[valid].mean(), valid.float().mean().item()

def decorrelation(crop, disease):
    if len(crop) < 2: return (crop.sum()+disease.sum())*0
    crop, disease = crop.float(), disease.float()
    def standardize(x): return (x-x.mean(0)) / x.std(0, unbiased=True).clamp_min(1e-6)
    covariance = standardize(crop).T @ standardize(disease) / (len(crop)-1)
    return covariance.square().mean()

def pair_scores(crop, disease, pairs):
    pairs = pairs.to(crop.device)
    return crop.log_softmax(1)[:, pairs[:, 0]] + disease.log_softmax(1)[:, pairs[:, 1]]

def zscore(x):
    return (x-x.mean(1, keepdim=True)) / x.std(1, unbiased=False, keepdim=True).clamp_min(1e-6)

def factorized(parts, pairs, weights):
    pair, crop, disease = parts
    if len(weights) != 3 or min(weights) < 0 or abs(sum(weights)-1) > 1e-6:
        raise ValueError('Three nonnegative scorer weights summing to one required.')
    pairs = pairs.to(pair.device)
    return weights[0]*zscore(pair)+weights[1]*zscore(crop)[:, pairs[:,0]]+weights[2]*zscore(disease)[:, pairs[:,1]]

RECIPE = dict(pair=1., primitive=1.)

def recipe_weights(recipe,config):
    if not recipe.get('aligned',True):return None
    if recipe.get('pair',0)==0:return [0.,.5,.5]
    if recipe.get('primitive',0)==0:return [1.,0.,0.]
    return config['score_grid'][0]

def score(output,pairs,family,recipe,weights=None):
    pair,crop,disease=output[:3]
    if family=='A' or not recipe.get('aligned',True):return pair_scores(crop,disease,pairs)
    return factorized((pair,crop,disease),pairs,weights)

def loss(output,labels,pairs,rows,family,recipe,config):
    _,crop,disease,uc,ud=output
    concepts=pairs.to(labels.device)[labels]
    zero=uc.sum()*0;ce_crop=ce_disease=pair_loss=zero
    if family=='A' or not recipe.get('aligned',True):
        ce_crop=F.cross_entropy(crop,concepts[:,0],label_smoothing=config['label_smoothing']);ce_disease=F.cross_entropy(disease,concepts[:,1],label_smoothing=config['label_smoothing'])
        value=ce_crop+ce_disease
    else:
        target_map={row:i for i,row in enumerate(rows)}
        try:targets=torch.tensor([target_map[int(i)] for i in labels],device=labels.device)
        except KeyError as error:raise ValueError('Proxy-held-out pair used as training target.') from error
        if recipe['pair']:pair_loss=F.cross_entropy(output[0][:,rows],targets,label_smoothing=config['label_smoothing'])
        if recipe['primitive']:
            ce_crop=F.cross_entropy(crop,concepts[:,0],label_smoothing=config['label_smoothing']);ce_disease=F.cross_entropy(disease,concepts[:,1],label_smoothing=config['label_smoothing'])
        value=recipe['pair']*pair_loss+recipe['primitive']*(ce_crop+ce_disease)
    sc,anchor=supcon(ud,concepts[:,0],concepts[:,1],config['supcon_temperature'],recipe.get('cross_crop',True))
    dis=decorrelation(uc,ud)
    value=value+recipe.get('sc',0.)*sc+recipe.get('dis',0.)*dis
    return value,dict(crop_ce=float(ce_crop.detach()),disease_ce=float(ce_disease.detach()),pair_ce=float(pair_loss.detach()),
        supcon=float(sc.detach()),decorrelation=float(dis.detach()),valid_anchor_fraction=anchor)

def make_model(pairs, names, metadata=None):
    cfg.set_seed(cfg.SEED)
    recipe = RECIPE
    metadata = metadata or {}
    backbone = Dino(HP, cfg.DEVICE, allow_download=True,
                    revision=metadata.get('dino', {}).get('revision'))
    backbone.policy(0, False)
    text = None; banks = None; prompts = None
    if FAMILY == 'B' and recipe.get('aligned', True):
        if recipe.get('semantic', True):
            text = TextOnly(HP, cfg.DEVICE, allow_download=True,
                            revision=(metadata.get('text') or {}).get('revision'))
            banks, prompts = text_banks(text, pairs, names, HP, recipe.get('ensemble', False))
        else:
            banks = nonsemantic_banks(pairs, cfg.DEVICE)
    torch.manual_seed(cfg.SEED)
    model = Model(backbone, FAMILY, HP, 512, banks).to(cfg.DEVICE)
    if recipe.get('original_baseline'):
        model.requires_grad_(False); model.original_baseline = True
        model.backbone.crop.head.requires_grad_(True)
        model.backbone.disease.head.requires_grad_(True)
    elif FAMILY == 'B' and recipe.get('aligned', True):
        model.crop_head.requires_grad_(False); model.disease_head.requires_grad_(False)
        if not recipe['pair']: model.pair_projection.requires_grad_(False)
        if not recipe['primitive']:
            model.crop_projection.requires_grad_(False)
            if not recipe.get('sc', 0): model.disease_projection.requires_grad_(False)
    torch.set_rng_state(backbone.augmentation_rng_state)
    return model, text, prompts, dict(dino=backbone.metadata, text=text.metadata if text else None)


def make_predict(model, candidate_pairs, recipe):
    weights = recipe_weights(recipe, HP) if FAMILY == 'B' else None
    @torch.no_grad()
    def predict(images):
        model.eval()
        values = model(images.to(cfg.DEVICE), aligned=FAMILY == 'B' and recipe.get('aligned', True))
        return score(values, candidate_pairs, FAMILY, recipe, weights)
    return predict


# 4. Training
Train one complete approach using the baseline training budget and common validation for checkpoint selection. No official test data is accessed.


In [ ]:
OUT.mkdir(parents=True, exist_ok=False)
cfg.set_seed(cfg.SEED)
train_loader, val_loader, _, _ = loaders()
for loader in (train_loader, val_loader): loader.pin_memory = False
model, text, prompts, metadata = make_model(pairs, concept_names)
recipe = RECIPE
crop_params = []; disease_params = []
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        (disease_params if 'disease' in name else crop_params).append(parameter)
optimizers = [torch.optim.AdamW(parameters, lr=lr, weight_decay=HP['weight_decay'])
              for parameters, lr in [(crop_params, HP['crop_lr']), (disease_params, HP['disease_lr'])] if parameters]
schedules = [torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS) for opt in optimizers]
history = []; best_val = -1.; checkpoint_path = OUT / (MODEL_NAME + '.pt')
for epoch in range(EPOCHS):
    model.train(); model.backbone.eval()
    total = 0.; samples = 0; anchor_fraction = 0.
    for images, _, _, labels in train_loader:
        for opt in optimizers: opt.zero_grad(set_to_none=True)
        values = model(images.to(cfg.DEVICE), aligned=FAMILY == 'B' and recipe.get('aligned', True))
        value, parts = loss(values, labels.to(cfg.DEVICE), pairs,
                            list(range(cfg.NUM_SEEN_PAIRS)), FAMILY, recipe, HP)
        if not torch.isfinite(value): raise RuntimeError('Nonfinite loss.')
        value.backward()
        if text is not None: text.assert_frozen()
        if any(p.grad is not None for p in model.backbone.parameters() if not p.requires_grad):
            raise RuntimeError('Frozen DINO received gradients.')
        gradients = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not gradients or not all(torch.isfinite(g).all() for g in gradients):
            raise RuntimeError('Invalid training gradients.')
        for opt in optimizers: opt.step()
        total += value.item() * len(labels); samples += len(labels)
        anchor_fraction += parts['valid_anchor_fraction'] * len(labels)
    for schedule in schedules: schedule.step()
    predict = make_predict(model, pairs, recipe)
    validation = val_accuracy(predict, val_loader)
    history.append(dict(epoch=epoch+1, loss=total/samples, val_top1=validation,
                        valid_crosscrop_anchor_fraction=anchor_fraction/samples))
    print(history[-1])
    if validation > best_val:
        best_val = validation
        torch.save(dict(model_name=MODEL_NAME,
            state_dict={k:v.detach().cpu() for k,v in model.state_dict().items()},
            hp=HP, debug=DEBUG, family=FAMILY, recipe=recipe,
            pairs=pairs, prompts=prompts, metadata=metadata, best_val=best_val,
            best_epoch=epoch+1, environment=environment), checkpoint_path)
del model, text, optimizers, schedules
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
selection = dict(model_name=MODEL_NAME, family=FAMILY, hp=HP, debug=DEBUG,
    recipe=RECIPE, checkpoint=checkpoint_path.name, best_val=best_val, history=history,
    train_images=train_loader.dataset.frame.image.tolist(),
    validation_images=val_loader.dataset.frame.image.tolist(),
    development_pairs=pairs.tolist(), criterion='common validation Top1; earliest epoch on ties')
with (OUT/'selection.json').open('x', encoding='utf8') as file: json.dump(selection, file, indent=2)
print('Selection frozen:', OUT/'selection.json')

# 5. Evaluation
Same common.evaluate as the old CLIP notebook. After ALL choices are frozen, enable RUN_FINAL_EVALUATION. Official333 candidates appear only here. The two extra component accuracies use the SAME clean unseen predictions, not robustness outputs; they describe unseen-combination test images, not genuinely unseen crop/disease classes.


In [ ]:
RUN_FINAL_EVALUATION = False  # Enable only after all model choices are frozen.
reports = []
if RUN_FINAL_EVALUATION:
    if DEBUG:
        raise ValueError('Official evaluation requires a full non-DEBUG run.')
    run_folder = OUT
    selection = json.loads((run_folder / 'selection.json').read_text(encoding='utf8'))
    checkpoint = torch.load(run_folder / selection['checkpoint'], map_location='cpu', weights_only=True)
    expected = dict(model_name=MODEL_NAME, family=FAMILY, debug=False, hp=HP, recipe=RECIPE)
    if any(record.get(key) != value for record in (selection, checkpoint)
           for key, value in expected.items()):
        raise ValueError('The selected run or checkpoint does not match the frozen settings.')
    if selection['development_pairs'] != pairs.tolist() or not torch.equal(checkpoint['pairs'], pairs):
        raise ValueError('Development candidates changed.')
    if (selection['train_images'] != train_loader.dataset.frame.image.tolist()
            or selection['validation_images'] != val_loader.dataset.frame.image.tolist()):
        raise ValueError('Common train/validation membership changed.')
    model, text, _, metadata = make_model(pairs, concept_names, checkpoint['metadata'])
    if metadata != checkpoint['metadata']:
        raise ValueError('Public pretrained model identity changed.')
    model.load_state_dict(checkpoint['state_dict'])
    model.eval().requires_grad_(False)
    # Only now open the official candidates and test loaders.
    final_pairs = get_pairs(final_evaluation=True)
    final_banks, _ = text_banks(text, final_pairs, concept_names, HP)
    model.install_text(final_banks)
    seen_loader, unseen_loader = get_final_test_loaders(
        HP['batch_size'], norm=HP['norm'], final_evaluation=True)
    sample_counts.update(seen=len(seen_loader.dataset), unseen=len(unseen_loader.dataset))
    predict_final = make_predict(model, final_pairs, RECIPE)
    # common.evaluate predicts clean seen, then clean unseen, then robustness.
    seen_count = sample_counts['seen']
    clean_end = seen_count + sample_counts['unseen']
    unseen_batches, processed = [], 0
    def predict(images):
        global processed
        scores = predict_final(images)
        if seen_count <= processed < clean_end:
            unseen_batches.append(scores.float().cpu())
        processed += len(images)
        return scores
    RESULTS = cfg.RESULTS_DIR / run_folder.name
    RESULTS.mkdir(parents=True, exist_ok=False)
    previous_results = cfg.RESULTS_DIR
    try:
        cfg.RESULTS_DIR = RESULTS
        result = evaluate(predict, seen_loader, unseen_loader, MODEL_NAME, HP, robust=True)
    finally:
        cfg.RESULTS_DIR = previous_results
    unseen_labels = torch.as_tensor(unseen_loader.dataset.frame.pair_row.to_numpy())
    crop_acc, disease_acc = concept_accuracy(torch.cat(unseen_batches), unseen_labels, final_evaluation=True)
    result.update(unseen_test_crop_acc=crop_acc, unseen_test_disease_acc=disease_acc,
                  debug=DEBUG, best_val=checkpoint['best_val'])
    reports.append(result)
    del predict_final, model, text, unseen_batches

# 6. Save
Same summary fields as the old CLIP notebook. Separate run folders prevent overwriting baselines. Keep checkpoints and result JSON/CSV/confusion files. One common seed does not establish statistical significance; formal results are not generated during authoring.


In [ ]:
if RUN_FINAL_EVALUATION:
    for result in reports:
        (RESULTS/(result['model']+'.json')).write_text(json.dumps(result, indent=2), encoding='utf8')
    summary = dict(selection=selection, results=reports, hp=HP,
                   environment=environment, samples=sample_counts)
    (RESULTS/(MODEL_NAME+'_comparison.json')).write_text(json.dumps(summary, indent=2), encoding='utf8')
    table = pd.DataFrame(reports)
    table.to_csv(RESULTS/(MODEL_NAME+'_comparison.csv'), index=False)
    display(table[['model','seen_top1','unseen_top1','harmonic_mean',
                   'unseen_test_crop_acc','unseen_test_disease_acc']])
    print('Checkpoints:', run_folder, 'Results:', RESULTS)
else:
    print('Save the complete checkpoint run folder after training:', OUT)
